# requirements_parser

## Example 1: names, specifiers and extras

In [ ]:
# Example 1: a plain requirement line becomes a Requirement with its name, extras and version specs
import requirements

reqs = list(requirements.parse("requests[security,socks] >= 2.8.1, < 3\nnumpy==2.3.4\n"))
by_name = {r.name: r for r in reqs}
assert sorted(by_name) == ["numpy", "requests"], sorted(by_name)
assert sorted(by_name["requests"].extras) == ["security", "socks"], by_name["requests"].extras  # order not kept
assert sorted(by_name["requests"].specs) == [("<", "3"), (">=", "2.8.1")], by_name["requests"].specs
assert by_name["numpy"].specs == [("==", "2.3.4")] and by_name["numpy"].specifier
print("Example 1:", [(r.name, sorted(r.extras), sorted(r.specs)) for r in reqs])

## Example 2: editable VCS requirements

In [ ]:
# Example 2: `-e git+...@rev#egg=name` is parsed into vcs, uri, revision and name, and marked editable
import requirements

line = "-e git+https://github.com/madpah/requirements-parser.git@v0.13.1#egg=requirements-parser"
(req,) = requirements.parse(line)
assert req.editable and req.vcs == "git", (req.editable, req.vcs)
assert req.revision == "v0.13.1" and req.name == "requirements-parser", (req.revision, req.name)
assert req.uri == "git+https://github.com/madpah/requirements-parser.git", req.uri  # keeps the vcs+ prefix
print("Example 2:", req.name, req.vcs, req.revision, req.uri)

## Example 3: a URL requirement with a hash fragment

In [ ]:
# Example 3: a direct URL keeps its hash; a `--hash=` continuation line on its own is skipped
import requirements

text = "https://example.org/pkgs/demo-1.0.tar.gz#egg=demo&sha256=" + "ab" * 32 + "\n--hash=sha256:" + "cd" * 32 + "\n"
reqs = list(requirements.parse(text))
assert len(reqs) == 1, reqs
(req,) = reqs
assert req.name == "demo" and req.hash_name == "sha256" and req.hash == "ab" * 32, (req.name, req.hash_name)
assert req.uri == "https://example.org/pkgs/demo-1.0.tar.gz", req.uri
print("Example 3:", req.name, req.hash_name, req.hash[:8] + "...", req.uri)

## Example 4: `-r` includes another requirements file

In [ ]:
# Example 4: parse() follows `-r other.txt` relative to the file being read
import pathlib
import tempfile

import requirements

with tempfile.TemporaryDirectory() as folder:
    base = pathlib.Path(folder)
    (base / "base.txt").write_text("click>=8\npackaging\n")
    (base / "dev.txt").write_text("-r base.txt\npytest\n")
    with open(base / "dev.txt") as handle:
        names = [r.name for r in requirements.parse(handle)]
assert names == ["click", "packaging", "pytest"], names
print("Example 4: dev.txt resolves to", names)

## Example 5: comments, blank lines, local paths and the dict-like view

In [ ]:
# Example 5: comment and blank lines yield nothing; a local path is a local_file; a Requirement reads like a dict
import requirements

text = "# a comment\n\n-e /opt/src/mylib\nattrs  # trailing comment\n"
reqs = list(requirements.parse(text))
assert [r.local_file for r in reqs] == [True, False], [r.local_file for r in reqs]
local, attrs = reqs
# A path is split like a file name: the last component is the name, the rest the path.
assert local.editable and (local.path, local.name) == ("/opt/src", "mylib"), (local.path, local.name)
assert attrs.name == "attrs" and attrs["name"] == "attrs" and "specs" in attrs.keys()
print("Example 5:", local.path, local.name, "| attrs keys:", sorted(attrs.keys())[:5], "...")